# Notebook 00 — Acquisition Massive de Données (Colab)
**Projet :** Système Intelligent de Détection de Spam et de Phishing  
**Auteur :** Nghogué Taptué Franck Roddier — 5GI, ENSPY

---

## Volume cible

| Source | Classe | Volume |
|--------|--------|--------|
| Enron (CMU) | Ham + Spam | ~517 000 |
| TREC 2007 | Spam + Ham | ~75 000 |
| SpamAssassin | Spam + Ham | ~9 000 |
| Nazario (monkey.org) | Phishing | ~9 500 |
| CEAS 2008 (Kaggle) | Spam + Ham | ~70 000 |
| **TOTAL** | | **~680 000** |

## Avant de lancer
1. `Runtime → Change runtime type → T4 GPU`  
2. Exécuter les cellules dans l'ordre  
3. Prévoir ~45 min pour tout télécharger (selon la connexion Colab)

## 0. Vérification GPU + montage Drive

In [9]:
import torch
if torch.cuda.is_available():
    print(f'✅ GPU : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('❌ Pas de GPU → Runtime → Change runtime type → T4 GPU')

✅ GPU : Tesla T4
   VRAM : 15.6 GB


In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/notebooks')
DATA_DIR    = PROJECT_DIR / 'data'
RAW_DIR     = PROJECT_DIR / 'raw'

for d in [PROJECT_DIR, DATA_DIR, RAW_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Lien symbolique pour que les notebooks accèdent à 'data/' directement
if not Path('/content/data').exists():
    os.symlink(str(DATA_DIR), '/content/data')

print(f'✅ Drive monté')
print(f'   Données sauvegardées dans : {DATA_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive monté
   Données sauvegardées dans : /content/drive/MyDrive/notebooks/data


In [11]:
%%capture
!pip install kaggle tqdm pandas numpy matplotlib seaborn
print('✅ Dépendances installées.')

## 1. Enron Email Corpus (~517 000 emails)

**Source officielle :** Carnegie Mellon University  
**URL :** `https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz`  
**Format :** Maildir (dossiers par employé)  
**Taille :** ~432 MB compressé, ~1.7 GB décompressé

Le corpus Enron ne contient que des hams à la base. Les labels spam/ham
viennent d'une annotation académique disponible séparément.

In [12]:
import subprocess

enron_tar = RAW_DIR / 'enron_mail.tar.gz'

if enron_tar.exists():
    print('✅ Archive Enron déjà présente dans Drive.')
else:
    print('Téléchargement Enron (~432 MB)...')
    result = subprocess.run([
        'wget', '-q', '--show-progress',
        'https://www.cs.cmu.edu/~enron/enron_mail_20150507.tar.gz',
        '-O', str(enron_tar)
    ], capture_output=True, text=True)
    if enron_tar.exists():
        size = enron_tar.stat().st_size / 1e6
        print(f'✅ Téléchargé : {size:.0f} MB')
    else:
        print('❌ Échec :', result.stderr[-300:])

✅ Archive Enron déjà présente dans Drive.


In [13]:
# Extraction dans /content (RAM, plus rapide que Drive pour l'extraction)
enron_extract = Path('/content/enron')

if not enron_extract.exists():
    print('Extraction (~3 min)...')
    result = subprocess.run(
        ['tar', '-xzf', str(enron_tar), '-C', '/content/'],
        capture_output=True, text=True
    )
    # Le tar crée un dossier maildir
    maildir = Path('/content/maildir')
    if maildir.exists():
        maildir.rename(enron_extract)
        print(f'✅ Extrait dans {enron_extract}')
    else:
        print('Dossiers extraits :', list(Path('/content').glob('enron*')))
else:
    print('✅ Déjà extrait.')

# Compter les employés et emails
employees = list(enron_extract.iterdir()) if enron_extract.exists() else []
print(f'Employés : {len(employees)}')

✅ Déjà extrait.
Employés : 150


In [14]:
# Téléchargement des labels spam Enron (annotation Metsis et al. 2006)
# Ces labels indiquent quels dossiers contiennent du spam
SPAM_FOLDERS = {
    'spam', 'junk', 'spam-final', 'spam-1', 'spam-2',
    'spam-3', 'spam-4', 'spam-5', 'trash', 'deleted items'
}
HAM_FOLDERS = {
    'inbox', 'sent', 'sent items', 'sent_items', 'all documents',
    '_sent_mail', 'discussion threads', 'notes inbox'
}

import email as email_lib
from tqdm import tqdm

def parse_maildir_email(filepath):
    """Lit un fichier email Maildir et retourne son texte brut."""
    try:
        raw  = Path(filepath).read_bytes()
        text = raw.decode('utf-8', errors='replace')
        msg  = email_lib.message_from_string(text)

        parts = []
        for header in ['Subject', 'From', 'To']:
            val = msg.get(header, '')
            if val: parts.append(f'{header}: {val}')

        if msg.is_multipart():
            for part in msg.walk():
                if part.get_content_type() == 'text/plain':
                    try:
                        charset = part.get_content_charset() or 'utf-8'
                        body = part.get_payload(decode=True).decode(charset, errors='replace')
                        parts.append(body)
                    except Exception:
                        pass
        else:
            try:
                charset = msg.get_content_charset() or 'utf-8'
                body = msg.get_payload(decode=True)
                if body:
                    parts.append(body.decode(charset, errors='replace'))
            except Exception:
                parts.append(str(msg.get_payload()))

        return '\n'.join(parts)
    except Exception:
        return ''


print('Parcours du corpus Enron...')
print('(Labellisation par nom de dossier — méthode Metsis et al. 2006)')

records_enron = []
MAX_PER_LABEL = 300_000   # Plafond par sécurité (RAM)
counts        = {'ham': 0, 'spam': 0}

for employee_dir in tqdm(employees, desc='Employés'):
    if not employee_dir.is_dir():
        continue
    for folder in employee_dir.iterdir():
        if not folder.is_dir():
            continue

        folder_name = folder.name.lower().replace('-', ' ')

        if any(s in folder_name for s in SPAM_FOLDERS):
            label = 'spam'
        elif any(h in folder_name for h in HAM_FOLDERS):
            label = 'ham'
        else:
            label = 'ham'  # Par défaut : ham

        if counts[label] >= MAX_PER_LABEL:
            continue

        for email_file in folder.iterdir():
            if email_file.is_file() and counts[label] < MAX_PER_LABEL:
                text = parse_maildir_email(email_file)
                if len(text.strip()) > 50:
                    records_enron.append({
                        'text': text, 'label': label, 'source': 'enron_maildir'
                    })
                    counts[label] += 1

import pandas as pd
df_enron = pd.DataFrame(records_enron)
print(f'\n✅ Enron chargé : {len(df_enron):,} emails')
print(df_enron['label'].value_counts())

Parcours du corpus Enron...
(Labellisation par nom de dossier — méthode Metsis et al. 2006)


Employés: 100%|██████████| 150/150 [01:12<00:00,  2.06it/s]



✅ Enron chargé : 300,029 emails
label
ham     300000
spam        29
Name: count, dtype: int64


## 2. TREC 2007 Spam Corpus (~75 000 emails)

**Source :** University of Waterloo  
**URL :** `https://plg.uwaterloo.ca/~gvcormac/treccorpus07/`  
**Format :** Trec (index + emails bruts)  
**Labels :** fournis dans un fichier `full/index` (spam / ham)

In [15]:
# ── CORRECTION COMPLÈTE ────────────────────────────────────────────
# Remplace TREC 2007 + CEAS 2008 + Ling-Spam par des sources Kaggle
# disponibles sans problème depuis Colab

import subprocess, zipfile, pandas as pd
from pathlib import Path

RAW_DIR  = Path('/content/drive/MyDrive/notebooks')
DATA_DIR = Path('/content/drive/MyDrive/notebooks/data')

# Variable bidon pour ne plus avoir l'erreur trec_tar
trec_tar = RAW_DIR / 'trec07p.tgz'
df_trec  = pd.DataFrame()   # DataFrame vide — sera ignoré à la fusion

records_kaggle = []

# ── 4 datasets Kaggle accessibles et volumineux ────────────────────
DATASETS = [
    # ~33 000 emails Enron labelisés proprement (ham/spam)
    ('gmspam/enron-spam-dataset',                    'enron2'),
    # ~11 000 phishing + ham
    ('subhajournal/phishingemails',                  'phishing_sub'),
    # ~5 500 SMS spam
    ('uciml/sms-spam-collection-dataset',            'sms_spam'),
    # ~4 500 phishing
    ('naserabdullahalam/phishing-email-dataset',     'phishing_nas'),
    # ~4 000 emails spam/ham variés
    ('balaka18/email-spam-classification-dataset-csv','spam_bal'),
    # ~2 500 Nigerian fraud (phishing)
    ('rtatman/fraudulent-email-corpus',              'fraud_nig'),
]

LABEL_MAP_FUNC = lambda s: (
    'phishing' if any(k in s for k in ['phish','fraud','scam','nigerian']) else
    'spam'     if 'spam' in s else
    'ham'      if any(k in s for k in ['ham','legit','safe','0']) else
    None
)

for dataset_id, tag in DATASETS:
    print(f'\n── {dataset_id} ──')
    extract_dir = Path(f'/content/raw_{tag}')

    # Téléchargement
    if not extract_dir.exists():
        r = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', dataset_id, '-p', '/content/'],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(f'  ⚠ Erreur : {r.stderr[-150:]}')
            continue

        # Décompression
        for zf in sorted(Path('/content').glob('*.zip')):
            try:
                with zipfile.ZipFile(zf) as z:
                    z.extractall(extract_dir)
                zf.unlink()
            except Exception:
                pass

    # Lecture de tous les CSV dans le dossier
    csvs = list(extract_dir.rglob('*.csv'))
    if not csvs:
        # Parfois c'est un .txt ou autre format
        txts = list(extract_dir.rglob('*.txt'))
        print(f'  Pas de CSV — fichiers trouvés : {[f.name for f in txts[:5]]}')
        continue

    for csv_path in csvs:
        try:
            # Essai utf-8 puis latin-1
            for enc in ['utf-8', 'latin-1', 'cp1252']:
                try:
                    df_raw = pd.read_csv(csv_path, encoding=enc, on_bad_lines='skip')
                    break
                except Exception:
                    continue

            cols  = [c.lower().strip() for c in df_raw.columns]
            print(f'  {csv_path.name} — {len(df_raw):,} lignes — colonnes: {list(df_raw.columns)}')

            # Détection automatique colonne texte
            text_col = next(
                (df_raw.columns[i] for i, c in enumerate(cols)
                 if any(k in c for k in ['text','body','message','email','content','v2'])),
                None
            )
            # Détection automatique colonne label
            label_col = next(
                (df_raw.columns[i] for i, c in enumerate(cols)
                 if any(k in c for k in ['label','class','type','spam','category','v1'])),
                None
            )

            if not text_col or not label_col:
                print(f'    ⚠ Colonnes non reconnues. Aperçu:')
                print(df_raw.head(2).to_string())
                continue

            count_before = len(records_kaggle)
            for _, row in df_raw.iterrows():
                raw_label = str(row[label_col]).lower().strip()
                label = LABEL_MAP_FUNC(raw_label)
                if label is None:
                    # Essai sur valeurs numériques (0=ham, 1=spam)
                    label = 'spam' if raw_label in ('1','spam') else 'ham'
                text = str(row[text_col]).strip()
                if len(text) > 20:
                    records_kaggle.append({
                        'text': text, 'label': label, 'source': f'kaggle_{tag}'
                    })
            added = len(records_kaggle) - count_before
            print(f'    → {added:,} emails ajoutés')

        except Exception as e:
            print(f'    ⚠ Erreur lecture : {e}')

df_kaggle = pd.DataFrame(records_kaggle)
print(f'\n{"="*50}')
print(f'Total Kaggle : {len(df_kaggle):,} emails')
print(df_kaggle['label'].value_counts())
print(df_kaggle['source'].value_counts())


── gmspam/enron-spam-dataset ──
  ⚠ Erreur : 

── subhajournal/phishingemails ──
  Phishing_Email.csv — 18,650 lignes — colonnes: ['Unnamed: 0', 'Email Text', 'Email Type']
    → 18,081 emails ajoutés

── uciml/sms-spam-collection-dataset ──
  spam.csv — 5,572 lignes — colonnes: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
    → 5,406 emails ajoutés

── naserabdullahalam/phishing-email-dataset ──
  Nazario.csv — 1,565 lignes — colonnes: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']
    → 1,555 emails ajoutés
  SpamAssasin.csv — 5,809 lignes — colonnes: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
    → 5,806 emails ajoutés
  Enron.csv — 29,767 lignes — colonnes: ['subject', 'body', 'label']
    → 29,629 emails ajoutés
  Ling.csv — 2,859 lignes — colonnes: ['subject', 'body', 'label']
    → 2,858 emails ajoutés
  CEAS_08.csv — 39,154 lignes — colonnes: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']
    → 39,145 em

In [16]:
import matplotlib.pyplot as plt

frames = []
for name, df in [
    ('Enron maildir',  df_enron),
    ('SpamAssassin',   df_sa),
    ('Nazario',        df_phishing),
    ('SMS Spam',       df_replacement),   # les 5510 déjà téléchargés
    ('Kaggle extra',   df_kaggle),        # les nouveaux
]:
    if len(df) > 0:
        frames.append(df)
        print(f'  {name:18s} : {len(df):>8,} emails')

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all[df_all['text'].str.strip().str.len() > 30].reset_index(drop=True)
df_all = df_all.drop_duplicates(subset=['text']).reset_index(drop=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n{"="*45}')
print(f'TOTAL : {len(df_all):,} emails')
print('='*45)
print(df_all['label'].value_counts())

NameError: name 'df_sa' is not defined

## 3. SpamAssassin Public Corpus (~9 000 emails)

**Source :** Apache SpamAssassin  
**Accès :** Direct wget, aucune inscription  
**Labels :** dans le nom de l'archive (spam / ham)

In [ ]:
import tarfile, io, requests

SA_ARCHIVES = {
    'spam': [
        'https://spamassassin.apache.org/old/publiccorpus/20050311_spam_2.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20030228_spam.tar.bz2',
    ],
    'ham': [
        'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20030228_easy_ham_2.tar.bz2',
        'https://spamassassin.apache.org/old/publiccorpus/20030228_hard_ham.tar.bz2',
    ],
}

records_sa = []

for label, urls in SA_ARCHIVES.items():
    for url in urls:
        archive_name = url.split('/')[-1]
        local_path   = RAW_DIR / archive_name

        # Téléchargement (si pas encore fait)
        if not local_path.exists():
            print(f'Téléchargement {archive_name}...')
            try:
                r = requests.get(url, timeout=60, stream=True)
                r.raise_for_status()
                with open(local_path, 'wb') as f:
                    for chunk in r.iter_content(8192):
                        f.write(chunk)
                print(f'  ✅ {local_path.stat().st_size/1e6:.1f} MB')
            except Exception as e:
                print(f'  ⚠ Erreur {archive_name}: {e}')
                continue
        else:
            print(f'✅ {archive_name} déjà dans Drive.')

        # Extraction et parsing
        try:
            with tarfile.open(local_path, 'r:bz2') as tar:
                members = [m for m in tar.getmembers() if m.isfile()]
                for member in tqdm(members, desc=f'{label}/{archive_name[:20]}', leave=False):
                    f = tar.extractfile(member)
                    if f:
                        text = f.read().decode('utf-8', errors='replace')
                        if len(text.strip()) > 30:
                            records_sa.append({
                                'text': text, 'label': label, 'source': 'spamassassin'
                            })
        except Exception as e:
            print(f'  ⚠ Erreur extraction {archive_name}: {e}')

df_sa = pd.DataFrame(records_sa)
print(f'\n✅ SpamAssassin : {len(df_sa):,} emails')
print(df_sa['label'].value_counts())

## 4. Nazario Phishing Corpus (~9 500 emails)

**Source officielle :** `https://monkey.org/~jose/phishing/`  
**Format :** Fichiers `.mbox` (format mailbox standard)  
**Méthode :** `wget` direct — aucun compte requis

In [ ]:
import mailbox

nazario_dir = RAW_DIR / 'nazario'
nazario_dir.mkdir(exist_ok=True)

# Liste des fichiers .mbox disponibles sur monkey.org
NAZARIO_FILES = [
    'https://monkey.org/~jose/phishing/phishing0.mbox',
    'https://monkey.org/~jose/phishing/phishing1.mbox',
    'https://monkey.org/~jose/phishing/phishing2.mbox',
    'https://monkey.org/~jose/phishing/phishing3.mbox',
    'https://monkey.org/~jose/phishing/phishing4.mbox',
    'https://monkey.org/~jose/phishing/phishing5.mbox',
]

print('Téléchargement du corpus Nazario depuis monkey.org...')

for url in NAZARIO_FILES:
    fname     = url.split('/')[-1]
    local_path = nazario_dir / fname

    if local_path.exists():
        print(f'  ✅ {fname} déjà dans Drive.')
        continue

    result = subprocess.run(
        ['wget', '-q', '-O', str(local_path), url],
        capture_output=True, text=True, timeout=120
    )

    if local_path.exists() and local_path.stat().st_size > 1000:
        print(f'  ✅ {fname} ({local_path.stat().st_size/1e6:.1f} MB)')
    else:
        print(f'  ⚠ {fname} — échec (taille={local_path.stat().st_size if local_path.exists() else 0})')
        if local_path.exists():
            local_path.unlink()  # Supprimer les fichiers vides

mbox_files = list(nazario_dir.glob('*.mbox'))
print(f'\nFichiers .mbox disponibles : {len(mbox_files)}')

In [ ]:
import mailbox

def parse_mbox_message(msg):
    """Extrait le texte d'un message mailbox."""
    parts = []
    for header in ['Subject', 'From']:
        val = msg.get(header, '')
        if val:
            parts.append(f'{header}: {val}')

    if msg.is_multipart():
        for part in msg.walk():
            if part.get_content_type() in ('text/plain', 'text/html'):
                try:
                    charset = part.get_content_charset() or 'utf-8'
                    payload = part.get_payload(decode=True)
                    if payload:
                        parts.append(payload.decode(charset, errors='replace'))
                except Exception:
                    pass
    else:
        try:
            payload = msg.get_payload(decode=True)
            if payload:
                charset = msg.get_content_charset() or 'utf-8'
                parts.append(payload.decode(charset, errors='replace'))
        except Exception:
            parts.append(str(msg.get_payload()))

    return '\n'.join(parts)


records_phishing = []

for mbox_path in mbox_files:
    print(f'Parsing {mbox_path.name}...')
    try:
        mbox = mailbox.mbox(str(mbox_path))
        count_before = len(records_phishing)
        for msg in mbox:
            text = parse_mbox_message(msg)
            if len(text.strip()) > 30:
                records_phishing.append({
                    'text': text, 'label': 'phishing', 'source': 'nazario'
                })
        added = len(records_phishing) - count_before
        print(f'  → {added} emails extraits')
    except Exception as e:
        print(f'  ⚠ Erreur parsing {mbox_path.name}: {e}')

df_phishing = pd.DataFrame(records_phishing)
print(f'\n✅ Nazario phishing : {len(df_phishing):,} emails')

## 5. Phishing supplémentaire via Kaggle

Si Nazario < 5000 emails, on complète avec des datasets Kaggle.
Nécessite une clé API Kaggle.

In [ ]:
import json

# ── Coller ici le contenu de ton kaggle.json ──────────────────────
KAGGLE_JSON = {
    "username": "TON_USERNAME",   # ← remplacer
    "key":      "TON_API_KEY"     # ← remplacer
}
# ─────────────────────────────────────────────────────────────────

kaggle_dir = Path.home() / '.config' / 'kaggle'
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_file = kaggle_dir / 'kaggle.json'
kaggle_file.write_text(json.dumps(KAGGLE_JSON))
kaggle_file.chmod(0o600)
print('✅ Clé Kaggle configurée.')

phishing_count = len(df_phishing)
print(f'Phishing actuel : {phishing_count}')
print('Si < 5000 → téléchargement complémentaire Kaggle recommandé.')

In [ ]:
import zipfile

# Datasets phishing sur Kaggle (du plus grand au plus petit)
KAGGLE_PHISHING_DATASETS = [
    # ~11 000 phishing + ham
    ('subhajournal/phishingemails',              'phishing_subha'),
    # ~4 500 phishing
    ('naserabdullahalam/phishing-email-dataset', 'phishing_naser'),
    # Dataset de fraude nigériane (~2 500 scam emails)
    ('rtatman/fraudulent-email-corpus',          'phishing_fraud'),
]

extra_phishing_records = []

for dataset_id, folder_name in KAGGLE_PHISHING_DATASETS:
    extract_path = Path(f'/content/raw_{folder_name}')

    if not extract_path.exists():
        print(f'Téléchargement {dataset_id}...')
        result = subprocess.run(
            ['kaggle', 'datasets', 'download', '-d', dataset_id,
             '-p', '/content/'],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(f'  ⚠ Erreur : {result.stderr[-200:]}')
            continue

        # Décompression
        for zf in Path('/content').glob('*.zip'):
            with zipfile.ZipFile(zf) as z:
                z.extractall(extract_path)
            zf.unlink()  # Supprimer le zip pour libérer de l'espace
        print(f'  ✅ Extrait dans {extract_path}')

    # Chargement des CSVs
    for csv_path in extract_path.rglob('*.csv'):
        try:
            df_raw = pd.read_csv(csv_path, on_bad_lines='skip', encoding='utf-8')
            cols   = [c.lower() for c in df_raw.columns]
            print(f'  {csv_path.name} — colonnes : {list(df_raw.columns)} ({len(df_raw):,} lignes)')

            # Détection automatique text + label
            text_col = next(
                (df_raw.columns[i] for i, c in enumerate(cols)
                 if any(k in c for k in ['text','body','message','email','content'])),
                None
            )
            label_col = next(
                (df_raw.columns[i] for i, c in enumerate(cols)
                 if any(k in c for k in ['label','class','type','spam','category'])),
                None
            )

            if text_col and label_col:
                for _, row in df_raw.iterrows():
                    raw_label = str(row[label_col]).lower()
                    if 'phish' in raw_label or 'fraud' in raw_label or 'scam' in raw_label:
                        mapped = 'phishing'
                    elif 'spam' in raw_label:
                        mapped = 'spam'
                    elif 'ham' in raw_label or 'legit' in raw_label or 'safe' in raw_label:
                        mapped = 'ham'
                    else:
                        continue

                    text = str(row[text_col])
                    if len(text.strip()) > 30:
                        extra_phishing_records.append({
                            'text': text, 'label': mapped,
                            'source': f'kaggle_{folder_name}'
                        })
            else:
                print(f'    ⚠ Colonnes non reconnues — aperçu :')
                print(df_raw.head(2).to_string())
        except Exception as e:
            print(f'    ⚠ Erreur lecture {csv_path.name}: {e}')

df_kaggle_extra = pd.DataFrame(extra_phishing_records)
if len(df_kaggle_extra) > 0:
    print(f'\n✅ Kaggle supplémentaire : {len(df_kaggle_extra):,} emails')
    print(df_kaggle_extra['label'].value_counts())

## 6. Fusion & bilan final

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fusion de toutes les sources
frames = []
for name, df in [
    ('Enron',        df_enron),
    ('TREC 2007',    df_trec),
    ('SpamAssassin', df_sa),
    ('Nazario',      df_phishing),
    ('Kaggle extra', df_kaggle_extra),
]:
    if len(df) > 0:
        frames.append(df)
        print(f'  {name:15s} : {len(df):>8,} emails')

df_all = pd.concat(frames, ignore_index=True)
df_all = df_all[df_all['text'].str.strip().str.len() > 50].reset_index(drop=True)
df_all = df_all.drop_duplicates(subset=['text']).reset_index(drop=True)
df_all = df_all.sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\n{'='*45}')
print(f'TOTAL (après déduplication) : {len(df_all):,} emails')
print(f'{'='*45}')
print(df_all['label'].value_counts())
print()

# Avertissement déséquilibre
counts = df_all['label'].value_counts()
for label, count in counts.items():
    bar = '█' * (count // 5000)
    print(f'  {label:10s} {count:>8,}  {bar}')

In [ ]:
# Calcul des features brutes
import re
from tqdm import tqdm
tqdm.pandas()

def extract_features(text):
    if not isinstance(text, str): return {}
    alpha = [c for c in text if c.isalpha()]
    return {
        'char_count':      len(text),
        'word_count':      len(text.split()),
        'url_count':       len(re.findall(r'https?://', text)),
        'exclamation':     text.count('!'),
        'uppercase_ratio': sum(1 for c in alpha if c.isupper()) / max(len(alpha), 1),
        'has_html':        int(bool(re.search(r'<[a-zA-Z][^>]*>', text))),
        'has_ip_url':      int(bool(re.search(r'https?://\d{1,3}\.\d{1,3}', text))),
        'dollar_count':    text.count('$'),
    }

print('Calcul des features brutes...')
feats    = df_all['text'].progress_apply(extract_features)
df_feats = pd.DataFrame(feats.tolist())
df_final = pd.concat([df_all[['text','label','source']], df_feats], axis=1)

# Export
out_path = DATA_DIR / 'emails_raw.csv'
df_final.to_csv(out_path, index=False, encoding='utf-8')

print(f'\n✅ Dataset exporté : {out_path}')
print(f'   {len(df_final):,} emails · {df_final.shape[1]} colonnes')
print(f'   Taille : {out_path.stat().st_size/1e6:.0f} MB')

In [ ]:
# Visualisation finale
CLASS_COLORS = {'ham':'#1D9E75', 'spam':'#D85A30', 'phishing':'#BA7517'}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution classes
counts = df_final['label'].value_counts()
bars   = axes[0].bar(counts.index, counts.values,
                     color=[CLASS_COLORS[l] for l in counts.index],
                     edgecolor='white', linewidth=2)
axes[0].set_title('Distribution des classes', fontweight='bold')
axes[0].set_ylabel('Nombre d\'emails')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2,
                 bar.get_height()+counts.max()*0.01,
                 f'{val:,}', ha='center', fontweight='bold', fontsize=10)

# Sources
src = df_final['source'].value_counts()
axes[1].pie(src.values, labels=src.index,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':1.5})
axes[1].set_title('Répartition par source', fontweight='bold')

# Longueur des emails
for label in df_final['label'].unique():
    vals = df_final[df_final['label']==label]['word_count'].clip(upper=500)
    axes[2].hist(vals, bins=50, alpha=0.6, label=label, color=CLASS_COLORS[label])
axes[2].set_title('Longueur (mots par email)', fontweight='bold')
axes[2].set_xlabel('Mots')
axes[2].legend()

plt.suptitle(f'Dataset final — {len(df_final):,} emails · {df_final["source"].nunique()} sources',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(DATA_DIR / '00_dataset_overview.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'\n✅ Tout est prêt. Lance maintenant : notebook_02_preprocessing.ipynb')